# KLA Image Restoration



NoisyLR + GT → paired dataset → train/validation split → fine-tune pretrained MambaIRv2 → calculate Charbonnier loss → update weights → validate with PSNR/SSIM → save best model → test on new images.

In [ ]:

from pathlib import Path

PROJECT_ROOT = Path("/home/oem/PROJS/Semicon26/MambaIR")

GT_DIR = PROJECT_ROOT / "datasets" / "test" / "HR"
NOISYLR_DIR = PROJECT_ROOT / "datasets" / "test" / "LR"


MODEL_CONFIG = PROJECT_ROOT / "options" / "test" / "mambairv2" / "test_MambaIRv2_SR_x2.yml"

PRETRAINED_CHECKPOINT = PROJECT_ROOT / "experiments" / "pretrained_models" / "mambairv2_classicSR_Base_x2.pth"

OUTPUT_DIR = PROJECT_ROOT / "experiments" / "kla_finetune"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCALE = 2
PATCH_SIZE_LR = 64
BATCH_SIZE = 1
NUM_WORKERS = 2

EPOCHS = 10
LR = 1e-5
WEIGHT_DECAY = 1e-4

VAL_RATIO = 0.1
SEED = 42

DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("GT_DIR:", GT_DIR)
print("NOISYLR_DIR:", NOISYLR_DIR)
print("MODEL_CONFIG:", MODEL_CONFIG)
print("CHECKPOINT:", PRETRAINED_CHECKPOINT)
print("DEVICE:", DEVICE)


In [ ]:


import os
import random
import yaml
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import sys
sys.path.insert(0, str(PROJECT_ROOT))

import basicsr.archs.mambairv2_arch
from basicsr.archs import build_network

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)


In [ ]:

# Discover and pair GT / NoisyLR files

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp", ".npy"}

def list_files(folder):
    folder = Path(folder)
    return sorted([p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

gt_files = list_files(GT_DIR)
lq_files = list_files(NOISYLR_DIR)

print(f"GT images:     {len(gt_files)}")
print(f"NoisyLR images:{len(lq_files)}")

def make_key(path, root):
    return str(path.relative_to(root).with_suffix("")).lower()

gt_by_rel = {make_key(p, GT_DIR): p for p in gt_files}
lq_by_rel = {make_key(p, NOISYLR_DIR): p for p in lq_files}

pairs = []

for key, gt in gt_by_rel.items():
    if key in lq_by_rel:
        pairs.append((lq_by_rel[key], gt))

if not pairs:
    gt_by_stem = {}
    lq_by_stem = {}

    for p in gt_files:
        gt_by_stem.setdefault(p.stem.lower(), []).append(p)
    for p in lq_files:
        lq_by_stem.setdefault(p.stem.lower(), []).append(p)

    common = sorted(set(gt_by_stem) & set(lq_by_stem))
    for stem in common:
        if len(gt_by_stem[stem]) == 1 and len(lq_by_stem[stem]) == 1:
            pairs.append((lq_by_stem[stem][0], gt_by_stem[stem][0]))

print("Matched pairs:", len(pairs))

if len(pairs) == 0:
    raise RuntimeError("No GT/NoisyLR pairs found. Check GT_DIR and NOISYLR_DIR.")

for lq, gt in pairs[:10]:
    print("LQ:", lq)
    print("GT:", gt)


In [ ]:

# Image loading utilities

def load_array(path):
    path = Path(path)

    if path.suffix.lower() == ".npy":
        arr = np.load(path)
    else:
        arr = np.array(Image.open(path))

    # HWC / HW -> float32
    if arr.ndim == 2:
        arr = arr[..., None]

    if arr.ndim != 3:
        raise ValueError(f"Unsupported image shape {arr.shape}: {path}")

    # Convert CHW -> HWC
    if arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
        arr = np.transpose(arr, (1, 2, 0))

    arr = arr.astype(np.float32)

    if arr.max() > 1.5:
        if arr.max() <= 255:
            arr /= 255.0
        elif arr.max() <= 65535:
            arr /= 65535.0

    if arr.shape[2] == 1:
        arr = np.repeat(arr, 3, axis=2)
    elif arr.shape[2] > 3:
        arr = arr[:, :, :3]

    return arr

def to_tensor(arr):
    return torch.from_numpy(np.ascontiguousarray(arr.transpose(2, 0, 1))).float()

for lq_path, gt_path in pairs[:5]:
    lq = load_array(lq_path)
    gt = load_array(gt_path)
    print(
        f"{lq_path.name}: "
        f"LQ={lq.shape}, range=({lq.min():.4f}, {lq.max():.4f}) | "
        f"GT={gt.shape}, range=({gt.min():.4f}, {gt.max():.4f})"
    )


In [ ]:


bad_shapes = []

for lq_path, gt_path in pairs:
    lq = load_array(lq_path)
    gt = load_array(gt_path)

    expected = (lq.shape[0] * SCALE, lq.shape[1] * SCALE)

    if gt.shape[:2] != expected:
        bad_shapes.append((lq_path.name, lq.shape[:2], gt.shape[:2], expected))

print("Pairs:", len(pairs))
print("Shape mismatches:", len(bad_shapes))

for item in bad_shapes[:10]:
    print(item)

if bad_shapes:
    print("\nWARNING: Not all pairs follow the configured SCALE =", SCALE)
    print("Check the dataset before training.")


In [ ]:


n_show = min(4, len(pairs))
fig, axes = plt.subplots(n_show, 2, figsize=(10, 4 * n_show))

if n_show == 1:
    axes = np.expand_dims(axes, axis=0)

for i, (lq_path, gt_path) in enumerate(pairs[:n_show]):
    lq = load_array(lq_path)
    gt = load_array(gt_path)

    axes[i, 0].imshow(np.clip(lq, 0, 1))
    axes[i, 0].set_title(f"NoisyLR: {lq.shape[:2]}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(np.clip(gt, 0, 1))
    axes[i, 1].set_title(f"GT: {gt.shape[:2]}")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:


rng = random.Random(SEED)
indices = list(range(len(pairs)))
rng.shuffle(indices)

n_val = max(1, int(len(indices) * VAL_RATIO))
val_indices = indices[:n_val]
train_indices = indices[n_val:]

train_pairs = [pairs[i] for i in train_indices]
val_pairs = [pairs[i] for i in val_indices]

print("Train pairs:", len(train_pairs))
print("Val pairs:  ", len(val_pairs))


In [ ]:


class PairedRestorationDataset(Dataset):
    def __init__(self, pairs, patch_size=64, scale=2, training=True):
        self.pairs = pairs
        self.patch_size = patch_size
        self.scale = scale
        self.training = training

    def __len__(self):
        return len(self.pairs)

    def random_crop(self, lq, gt):
        h, w = lq.shape[:2]
        ps = self.patch_size

        if h < ps or w < ps:
            raise ValueError(
                f"LQ image {(h,w)} is smaller than patch size {ps}. "
                "Reduce PATCH_SIZE_LR."
            )

        top = random.randint(0, h - ps)
        left = random.randint(0, w - ps)

        gt_top = top * self.scale
        gt_left = left * self.scale
        gt_ps = ps * self.scale

        lq = lq[top:top+ps, left:left+ps]
        gt = gt[gt_top:gt_top+gt_ps, gt_left:gt_left+gt_ps]

        return lq, gt

    def augment(self, lq, gt):
        if random.random() < 0.5:
            lq = np.flip(lq, axis=1).copy()
            gt = np.flip(gt, axis=1).copy()

        if random.random() < 0.5:
            lq = np.flip(lq, axis=0).copy()
            gt = np.flip(gt, axis=0).copy()

        if random.random() < 0.5:
            lq = np.rot90(lq, k=1).copy()
            gt = np.rot90(gt, k=1).copy()

        return lq, gt

    def __getitem__(self, idx):
        lq_path, gt_path = self.pairs[idx]

        lq = load_array(lq_path)
        gt = load_array(gt_path)

        if self.training:
            lq, gt = self.random_crop(lq, gt)
            lq, gt = self.augment(lq, gt)

        return {
            "lq": to_tensor(lq),
            "gt": to_tensor(gt),
            "lq_path": str(lq_path),
            "gt_path": str(gt_path),
        }

train_dataset = PairedRestorationDataset(
    train_pairs,
    patch_size=PATCH_SIZE_LR,
    scale=SCALE,
    training=True,
)

val_dataset = PairedRestorationDataset(
    val_pairs,
    patch_size=PATCH_SIZE_LR,
    scale=SCALE,
    training=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
)

sample = train_dataset[0]
print("LQ tensor:", sample["lq"].shape)
print("GT tensor:", sample["gt"].shape)


In [ ]:


with open(MODEL_CONFIG, "r") as f:
    opt = yaml.safe_load(f)

network_cfg = opt["network_g"]

print("Network configuration:")
print(yaml.dump(network_cfg, sort_keys=False))

model = build_network(network_cfg).to(DEVICE)

checkpoint = torch.load(PRETRAINED_CHECKPOINT, map_location="cpu")

if "params_ema" in checkpoint:
    state_dict = checkpoint["params_ema"]
elif "params" in checkpoint:
    state_dict = checkpoint["params"]
elif "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint

missing, unexpected = model.load_state_dict(state_dict, strict=False)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

if missing:
    print("First missing:", missing[:10])
if unexpected:
    print("First unexpected:", unexpected[:10])

model.train()


In [ ]:


class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        return torch.mean(torch.sqrt((pred - target) ** 2 + self.eps ** 2))

criterion = CharbonnierLoss()

def batch_psnr(pred, target, max_val=1.0):
    pred = torch.clamp(pred, 0, 1)
    target = torch.clamp(target, 0, 1)

    mse = F.mse_loss(pred, target, reduction="none")
    mse = mse.flatten(1).mean(dim=1)
    psnr = 10 * torch.log10(max_val ** 2 / torch.clamp(mse, min=1e-10))
    return psnr.mean().item()

def ssim_simple(pred, target, window_size=11, C1=0.01**2, C2=0.03**2):
    pred = torch.clamp(pred, 0, 1)
    target = torch.clamp(target, 0, 1)

    mu_x = F.avg_pool2d(pred, window_size, stride=1, padding=window_size//2)
    mu_y = F.avg_pool2d(target, window_size, stride=1, padding=window_size//2)

    sigma_x = F.avg_pool2d(pred * pred, window_size, stride=1, padding=window_size//2) - mu_x**2
    sigma_y = F.avg_pool2d(target * target, window_size, stride=1, padding=window_size//2) - mu_y**2
    sigma_xy = F.avg_pool2d(pred * target, window_size, stride=1, padding=window_size//2) - mu_x * mu_y

    ssim = ((2 * mu_x * mu_y + C1) * (2 * sigma_xy + C2)) / (
        (mu_x**2 + mu_y**2 + C1) * (sigma_x + sigma_y + C2) + 1e-8
    )

    return ssim.mean().item()


In [ ]:


batch = next(iter(train_loader))
lq = batch["lq"].to(DEVICE, non_blocking=True)
gt = batch["gt"].to(DEVICE, non_blocking=True)

with torch.no_grad():
    pred = model(lq)

print("Input :", tuple(lq.shape))
print("Output:", tuple(pred.shape))
print("GT    :", tuple(gt.shape))

assert pred.shape == gt.shape, (
    f"Model output {pred.shape} does not match GT {gt.shape}. "
    "Check SCALE and MODEL_CONFIG."
)


In [ ]:


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
)

# AMP is useful on modern NVIDIA GPUs. Disable it automatically on CPU.
use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

print("AMP:", use_amp)


In [ ]:


history = {
    "train_loss": [],
    "val_loss": [],
    "val_psnr": [],
    "val_ssim": [],
}

best_psnr = -float("inf")
best_path = OUTPUT_DIR / "mambairv2_kla_best.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for batch in train_loader:
        lq = batch["lq"].to(DEVICE, non_blocking=True)
        gt = batch["gt"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            pred = model(lq)
            loss = criterion(pred, gt)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    train_loss = running_loss / max(1, len(train_loader))

    model.eval()
    val_loss = 0.0
    psnr_values = []
    ssim_values = []

    with torch.no_grad():
        for batch in val_loader:
            lq = batch["lq"].to(DEVICE, non_blocking=True)
            gt = batch["gt"].to(DEVICE, non_blocking=True)


            with torch.cuda.amp.autocast(enabled=use_amp):
                pred = model(lq)
                loss = criterion(pred, gt)

            val_loss += loss.item()
            psnr_values.append(batch_psnr(pred, gt))
            ssim_values.append(ssim_simple(pred, gt))

    val_loss /= max(1, len(val_loader))
    val_psnr = float(np.mean(psnr_values))
    val_ssim = float(np.mean(ssim_values))

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_psnr"].append(val_psnr)
    history["val_ssim"].append(val_ssim)

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"train={train_loss:.6f} | "
        f"val={val_loss:.6f} | "
        f"PSNR={val_psnr:.3f} | "
        f"SSIM={val_ssim:.4f} | "
        f"lr={optimizer.param_groups[0]['lr']:.2e}"
    )

    if val_psnr > best_psnr:
        best_psnr = val_psnr

        torch.save(
            {
                "params": model.state_dict(),
                "epoch": epoch,
                "val_psnr": val_psnr,
                "val_ssim": val_ssim,
                "config": network_cfg,
            },
            best_path,
        )

        print("  Saved best checkpoint:", best_path)

print("Best PSNR:", best_psnr)


In [ ]:


fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history["val_psnr"])
axes[1].set_title("Validation PSNR")

axes[2].plot(history["val_ssim"])
axes[2].set_title("Validation SSIM")

plt.tight_layout()
plt.show()


In [ ]:


best_ckpt = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(best_ckpt["params"], strict=True)
model.eval()

print("Loaded:", best_path)
print("Best epoch:", best_ckpt.get("epoch"))
print("Best PSNR:", best_ckpt.get("val_psnr"))
print("Best SSIM:", best_ckpt.get("val_ssim"))


In [ ]:


n_show = min(4, len(val_dataset))
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))

if n_show == 1:
    axes = np.expand_dims(axes, axis=0)

with torch.no_grad():
    for i in range(n_show):
        sample = val_dataset[i]

        lq = sample["lq"].unsqueeze(0).to(DEVICE)
        gt = sample["gt"].unsqueeze(0).to(DEVICE)

        with torch.cuda.amp.autocast(enabled=use_amp):
            pred = model(lq)

        pred = pred.clamp(0, 1)[0].cpu().permute(1, 2, 0).numpy()
        lq_img = lq[0].cpu().permute(1, 2, 0).numpy()
        gt_img = gt[0].cpu().permute(1, 2, 0).numpy()

        axes[i, 0].imshow(np.clip(lq_img, 0, 1))
        axes[i, 0].set_title("NoisyLR")

        axes[i, 1].imshow(pred)
        axes[i, 1].set_title("Restored")

        axes[i, 2].imshow(np.clip(gt_img, 0, 1))
        axes[i, 2].set_title("GT")

        for j in range(3):
            axes[i, j].axis("off")

plt.tight_layout()
plt.show()


In [ ]:


@torch.no_grad()
def restore_tensor_tiled(model, image, tile_size=64, overlap=16, scale=2):
    """
    image: [1,3,H,W] in the same range expected by MambaIRv2.
    Returns: [1,3,H*scale,W*scale]
    """
    assert image.ndim == 4 and image.shape[0] == 1

    _, _, h, w = image.shape
    stride = tile_size - overlap
    assert stride > 0

    pad_h = (tile_size - h % tile_size) % tile_size
    pad_w = (tile_size - w % tile_size) % tile_size

    padded = F.pad(image, (0, pad_w, 0, pad_h), mode="reflect")
    ph, pw = padded.shape[-2:]

    out = torch.zeros(
        (1, 3, ph * scale, pw * scale),
        device=image.device,
        dtype=torch.float32,
    )
    weight = torch.zeros_like(out)

    ys = list(range(0, max(1, ph - tile_size + 1), stride))
    xs = list(range(0, max(1, pw - tile_size + 1), stride))

    if ys[-1] != ph - tile_size:
        ys.append(ph - tile_size)
    if xs[-1] != pw - tile_size:
        xs.append(pw - tile_size)

    for y in ys:
        for x in xs:
            tile = padded[:, :, y:y+tile_size, x:x+tile_size]

            with torch.cuda.amp.autocast(enabled=use_amp):
                tile_out = model(tile)

            tile_out = tile_out.float()

            oy = y * scale
            ox = x * scale
            oh = tile_size * scale
            ow = tile_size * scale

            out[:, :, oy:oy+oh, ox:ox+ow] += tile_out
            weight[:, :, oy:oy+oh, ox:ox+ow] += 1.0

    out = out / torch.clamp(weight, min=1.0)

    return out[:, :, :h*scale, :w*scale].clamp(0, 1)


In [ ]:


model.eval()

n_show = min(3, len(val_pairs))

fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))

if n_show == 1:
    axes = np.expand_dims(axes, axis=0)

for i, (lq_path, gt_path) in enumerate(val_pairs[:n_show]):
    lq_arr = load_array(lq_path)
    gt_arr = load_array(gt_path)

    lq = to_tensor(lq_arr).unsqueeze(0).to(DEVICE)
    gt = to_tensor(gt_arr).unsqueeze(0).to(DEVICE)

    pred = restore_tensor_tiled(
        model,
        lq,
        tile_size=PATCH_SIZE_LR,
        overlap=16,
        scale=SCALE,
    )

    pred_img = pred[0].cpu().permute(1, 2, 0).numpy()

    axes[i, 0].imshow(np.clip(lq_arr, 0, 1))
    axes[i, 0].set_title(f"Input {lq_arr.shape[:2]}")

    axes[i, 1].imshow(pred_img)
    axes[i, 1].set_title(f"Restored {pred_img.shape[:2]}")

    axes[i, 2].imshow(np.clip(gt_arr, 0, 1))
    axes[i, 2].set_title(f"GT {gt_arr.shape[:2]}")

    for j in range(3):
        axes[i, j].axis("off")

plt.tight_layout()
plt.show()


In [ ]:


RESTORE_DIR = OUTPUT_DIR / "restored_examples"
RESTORE_DIR.mkdir(parents=True, exist_ok=True)

for lq_path, gt_path in val_pairs[:5]:
    lq_arr = load_array(lq_path)
    lq = to_tensor(lq_arr).unsqueeze(0).to(DEVICE)

    pred = restore_tensor_tiled(
        model, lq,
        tile_size=PATCH_SIZE_LR,
        overlap=16,
        scale=SCALE,
    )

    pred_arr = (
        pred[0]
        .cpu()
        .permute(1, 2, 0)
        .numpy()
    )

    pred_uint8 = (pred_arr * 255.0).round().clip(0, 255).astype(np.uint8)
    Image.fromarray(pred_uint8).save(RESTORE_DIR / f"{lq_path.stem}_restored.png")

print("Saved examples to:", RESTORE_DIR)
